In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!find "/content/drive/MyDrive/thermal_vision_project/datasets/LLVIP_YOLO/train/images" | wc -l

122


In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/thermal_vision_project/datasets/coco_subset"

In [ ]:
import os
import json
from collections import Counter
from PIL import Image

# =========================
# CONFIG
# =========================

DATASET_ROOT = "/content/drive/MyDrive/thermal_vision_project/datasets/coco_subset"

TRAIN_IMAGES = os.path.join(DATASET_ROOT, "images/train")
VAL_IMAGES = os.path.join(DATASET_ROOT, "images/val")

TRAIN_LABELS = os.path.join(DATASET_ROOT, "labels/train")
VAL_LABELS = os.path.join(DATASET_ROOT, "labels/val")

TRAIN_JSON = os.path.join(
    DATASET_ROOT,
    "annotations/instances_train2017.json"
)

VAL_JSON = os.path.join(
    DATASET_ROOT,
    "annotations/instances_val2017.json"
)

VALID_EXTENSIONS = [".jpg", ".jpeg", ".png"]


# =========================
# HELPERS
# =========================

def list_images(folder):
    files = []

    for f in os.listdir(folder):
        ext = os.path.splitext(f)[1].lower()

        if ext in VALID_EXTENSIONS:
            files.append(f)

    return files


def list_labels(folder):
    return [
        f for f in os.listdir(folder)
        if f.endswith(".txt")
    ]


def verify_images(folder, image_files):
    corrupted = []

    for img_name in image_files:
        path = os.path.join(folder, img_name)

        try:
            img = Image.open(path)
            img.verify()

        except Exception:
            corrupted.append(img_name)

    return corrupted


def verify_json(json_path):
    try:
        with open(json_path, "r") as f:
            data = json.load(f)

        required_keys = ["images", "annotations", "categories"]

        for key in required_keys:
            if key not in data:
                print(f"❌ Missing key: {key}")

        return data

    except Exception as e:
        print(f"❌ JSON ERROR: {e}")
        return None


def verify_person_only(data):
    invalid = []

    for ann in data["annotations"]:
        if ann["category_id"] != 1:
            invalid.append(ann["category_id"])

    return invalid


def verify_label_matching(images, labels):
    image_stems = set(os.path.splitext(f)[0] for f in images)
    label_stems = set(os.path.splitext(f)[0] for f in labels)

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems

    return missing_labels, missing_images


def count_annotations(data):
    counter = Counter()

    for ann in data["annotations"]:
        counter[ann["image_id"]] += 1

    return counter


# =========================
# MAIN
# =========================

print("\n========================")
print("DATASET VERIFICATION")
print("========================\n")


# -------------------------
# LIST FILES
# -------------------------

train_images = list_images(TRAIN_IMAGES)
val_images = list_images(VAL_IMAGES)

train_labels = list_labels(TRAIN_LABELS)
val_labels = list_labels(VAL_LABELS)

print(f"✅ Train images: {len(train_images)}")
print(f"✅ Val images:   {len(val_images)}")

print(f"✅ Train labels: {len(train_labels)}")
print(f"✅ Val labels:   {len(val_labels)}")


# -------------------------
# VERIFY IMAGES
# -------------------------

print("\nChecking image integrity...")

bad_train = verify_images(TRAIN_IMAGES, train_images)
bad_val = verify_images(VAL_IMAGES, val_images)

print(f"✅ Corrupted train images: {len(bad_train)}")
print(f"✅ Corrupted val images:   {len(bad_val)}")


# -------------------------
# VERIFY JSON
# -------------------------

print("\nChecking annotation JSON...")

train_data = verify_json(TRAIN_JSON)
val_data = verify_json(VAL_JSON)

if train_data and val_data:
    print("✅ JSON files loaded successfully")


# -------------------------
# VERIFY PERSON ONLY
# -------------------------

print("\nChecking category filtering...")

invalid_train = verify_person_only(train_data)
invalid_val = verify_person_only(val_data)

print(f"✅ Invalid train categories: {len(invalid_train)}")
print(f"✅ Invalid val categories:   {len(invalid_val)}")


# -------------------------
# VERIFY LABEL MATCHING
# -------------------------

print("\nChecking image-label consistency...")

missing_train_labels, missing_train_images = verify_label_matching(
    train_images,
    train_labels
)

missing_val_labels, missing_val_images = verify_label_matching(
    val_images,
    val_labels
)

print(f"✅ Missing train labels: {len(missing_train_labels)}")
print(f"✅ Missing val labels:   {len(missing_val_labels)}")

print(f"✅ Labels without train images: {len(missing_train_images)}")
print(f"✅ Labels without val images:   {len(missing_val_images)}")


# -------------------------
# ANNOTATION STATS
# -------------------------

print("\nComputing annotation statistics...")

train_counter = count_annotations(train_data)
val_counter = count_annotations(val_data)

avg_train = sum(train_counter.values()) / len(train_counter)
avg_val = sum(val_counter.values()) / len(val_counter)

print(f"✅ Train annotations: {sum(train_counter.values())}")
print(f"✅ Val annotations:   {sum(val_counter.values())}")

print(f"✅ Avg annotations/image (train): {avg_train:.2f}")
print(f"✅ Avg annotations/image (val):   {avg_val:.2f}")


# -------------------------
# DONE
# -------------------------

print("\n========================")
print("VERIFICATION COMPLETE")
print("========================\n")


DATASET VERIFICATION

✅ Train images: 5000
✅ Val images:   1000
✅ Train labels: 5000
✅ Val labels:   1001

Checking image integrity...


KeyboardInterrupt: 

In [ ]:
import os

# =========================
# CONFIG
# =========================

DATASET_ROOT = "/content/drive/MyDrive/thermal_vision_project/datasets/coco_subset"

TRAIN_LABELS = os.path.join(DATASET_ROOT, "labels/train")
VAL_LABELS = os.path.join(DATASET_ROOT, "labels/val")

# =========================
# VALIDATION
# =========================

def validate_label_file(label_path):

    errors = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line_num, line in enumerate(lines):

        parts = line.strip().split()

        # Must contain exactly 5 values
        if len(parts) != 5:
            errors.append(
                f"Line {line_num+1}: Expected 5 values"
            )
            continue

        try:
            class_id = int(parts[0])

            x = float(parts[1])
            y = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

        except:
            errors.append(
                f"Line {line_num+1}: Non-numeric values"
            )
            continue

        # Only person class allowed
        if class_id != 0:
            errors.append(
                f"Line {line_num+1}: Invalid class_id {class_id}"
            )

        # Normalization checks
        values = [x, y, w, h]

        for v in values:
            if v < 0 or v > 1:
                errors.append(
                    f"Line {line_num+1}: Value out of range [0,1]"
                )

        # Width/height checks
        if w <= 0 or h <= 0:
            errors.append(
                f"Line {line_num+1}: Invalid width/height"
            )

    return errors


# =========================
# RUN CHECKS
# =========================

def check_folder(folder):

    print(f"\nChecking: {folder}")

    label_files = [
        f for f in os.listdir(folder)
        if f.endswith(".txt")
    ]

    broken_files = 0
    total_boxes = 0

    for file_name in label_files:

        path = os.path.join(folder, file_name)

        errors = validate_label_file(path)

        with open(path, "r") as f:
            total_boxes += len(f.readlines())

        if errors:

            broken_files += 1

            print(f"\n❌ Errors in {file_name}")

            for err in errors[:5]:
                print("   ", err)

    print(f"\n✅ Total label files: {len(label_files)}")
    print(f"✅ Total boxes: {total_boxes}")
    print(f"✅ Broken files: {broken_files}")


# =========================
# MAIN
# =========================

print("\n========================")
print("YOLO LABEL VERIFICATION")
print("========================")

check_folder(TRAIN_LABELS)
check_folder(VAL_LABELS)

print("\n========================")
print("CHECK COMPLETE")
print("========================")


YOLO LABEL VERIFICATION

Checking: /content/drive/MyDrive/thermal_vision_project/datasets/coco_subset/labels/train

✅ Total label files: 5000
✅ Total boxes: 20253
✅ Broken files: 0

Checking: /content/drive/MyDrive/thermal_vision_project/datasets/coco_subset/labels/val

✅ Total label files: 1001
✅ Total boxes: 8199
✅ Broken files: 0

CHECK COMPLETE


In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================

DATASET_ROOT = "/content/drive/MyDrive/thermal_vision_project/datasets/coco_subset"

IMAGE_DIR = os.path.join(DATASET_ROOT, "images/train")
LABEL_DIR = os.path.join(DATASET_ROOT, "labels/train")

NUM_SAMPLES = 20

# =========================
# GET RANDOM IMAGES
# =========================

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.endswith((".jpg", ".png", ".jpeg"))
]

samples = random.sample(
    image_files,
    min(NUM_SAMPLES, len(image_files))
)

# =========================
# VISUALIZE
# =========================

for image_name in samples:

    image_path = os.path.join(IMAGE_DIR, image_name)

    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_path = os.path.join(LABEL_DIR, label_name)

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h_img, w_img, _ = image.shape

    if os.path.exists(label_path):

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:

            cls, x, y, w, h = map(float, line.strip().split())

            # Convert YOLO → pixel coords
            x1 = int((x - w / 2) * w_img)
            y1 = int((y - h / 2) * h_img)

            x2 = int((x + w / 2) * w_img)
            y2 = int((y + h / 2) * h_img)

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

    plt.figure(figsize=(8,8))
    plt.imshow(image)
    plt.title(image_name)
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.